In [1]:
import os
import gc
import zarr
import yaml
import json
import numba
import numpy as np
import polars as pl
import pandas as pd
from tqdm import tqdm

In [2]:
anngeno_path = '/home/dnanexus/data_dir/genebass_1e6_coding_variants.ag'
eur_samples_path = '/home/dnanexus/data_dir/unrelated_cauc_samples_3rd_degree.csv'
# olink_path = '/home/dnanexus/data_dir/olink/protrider_lite_output/log2fc.csv'
olink_path = "/home/dnanexus/data_dir/olink/olink_corrected_rint_90_pcs.parquet"

In [3]:
sample_ids = zarr.open(f'{anngeno_path}/zarr_store/samples', mode='r')[:]
var_ids = pl.read_parquet(f'{anngeno_path}/variant_metadata.parquet', columns=['id'])['id'].to_numpy()
geno = zarr.open(f'{anngeno_path}/zarr_store/genotypes', mode='r')

eur_samples = pl.read_csv(eur_samples_path).rename({'eid': 'individual'}).with_columns(
    pl.col("individual").cast(pl.Utf8)
)['individual'].to_list()

/home/dnanexus/deeprvat2-env/lib/python3.11/site-packages/zarr/codecs/vlen_utf8.py:44: UserWarning: The codec `vlen-utf8` is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  return cls(**configuration_parsed)


In [4]:
olink_df = pl.read_parquet(olink_path).rename({'sample': 'individual'}).with_columns(
    pl.col("individual").cast(pl.Utf8)
).filter(
    pl.col("individual").is_in(eur_samples)
).fill_nan(None)

unique_phenotypes = [pheno for pheno in olink_df.columns if pheno != 'individual']

olink_melt = (
    olink_df.unpivot(
        index=['individual'],
        on=unique_phenotypes,
        variable_name='phenotype',
        value_name='pheno_value',
    )
    .with_columns(
        phenotype = pl.col('phenotype') + '_olink'
    )
)

olink_melt

individual,phenotype,pheno_value
str,str,f64
"""1000107""","""ENSG00000120053_olink""",-0.592022
"""1001193""","""ENSG00000120053_olink""",0.061036
"""1001694""","""ENSG00000120053_olink""",0.228713
"""1001715""","""ENSG00000120053_olink""",0.674837
"""1002486""","""ENSG00000120053_olink""",1.153924
…,…,…
"""6019565""","""ENSG00000066468_olink""",0.72708
"""6019855""","""ENSG00000066468_olink""",-0.327369
"""6021528""","""ENSG00000066468_olink""",0.991073


In [7]:
# Optimized approach to find indices of olink_samples in sample_ids
# Step A: Create a lookup dictionary mapping each ID in the large array to its original index. This takes O(N) time, where N is the size of sample_ids.
olink_sids = olink_df['individual'].to_numpy()
sample_id_to_index = {sid: i for i, sid in enumerate(sample_ids)}

# Step B: Iterate through the smaller array and find the index for each element if it exists in our lookup dictionary. This takes O(M) time, where M is the size of olink_sids.
found_indices = []
for sid in olink_sids:
    if sid in sample_id_to_index:
        found_indices.append(sample_id_to_index[sid])
print(f"Found {len(found_indices)} matching IDs.")

# Step C: Convert the list of indices to a NumPy array and sort it.
# Sorting ensures the output is identical to the original np.where approach, which returns indices in ascending order.
olink_indices = np.sort(found_indices)
olink_indices

Found 40398 matching IDs.


array([    17,     23,     45, ..., 490517, 490532, 490535],
      shape=(40398,))

In [8]:
sample_ids[olink_indices]

array(['5102203', '3741733', '4072771', ..., '5141119', '5145210',
       '5733954'], shape=(40398,), dtype=StringDType())

In [9]:
len(set(olink_lazy['individual'].to_list()).intersection(set(sample_ids[olink_indices])))

NameError: name 'olink_lazy' is not defined

In [10]:
@numba.njit(parallel=True, fastmath=True)
def _fast_clip_and_sum_allels(arr):
    # Get the shape of the input array
    # Using specific dimensions for clarity with this problem
    n_samples, n_variants, _ = arr.shape
    
    # The sum of two positive int8s can be up to 254. 
    # An int16 is a safe and fast output type.
    output = np.empty((n_samples, n_variants), dtype=np.int8)
    
    # Numba's prange enables automatic parallelization across all your CPU cores
    for i in numba.prange(n_samples):
        for j in range(n_variants):
            # Read two values, perform logic, write one value.
            # This is the "fused" operation.
            val1 = arr[i, j, 0]
            val2 = arr[i, j, 1]
            
            s = 0
            # Since input is int8, this check is faster than max(0, val)
            if val1 > 0:
                s += val1
            if val2 > 0:
                s += val2
            
            output[i, j] = s
            
    return output

def process_genotype_chunk(
    geno: np.array, 
    var_ids: np.array, 
    sample_list: np.array,
    melted_pheno_df: pl.LazyFrame,
    homozygous: bool = False,
    debug: bool = False,
) -> pl.LazyFrame:
    """
    Extract genotypes for a specific gene and return as lazy DataFrame
    """
    geno_clipped = _fast_clip_and_sum_allels(geno)
    # Find heterozygous genotypes (genotype == 1)
    rows, cols = np.where(geno_clipped == 1)
    geno_melt = pl.DataFrame({
        'id': var_ids[rows],
        'individual': sample_list[cols],
        'genotype': 1
    })
    
    # Find homozygous genotypes (genotype == 2)
    if homozygous:
        rows, cols = np.where(geno_clipped == 2)
        hom = pl.DataFrame({
            'id': var_ids[rows],
            'individual': sample_list[cols],
            'genotype': 2
        })
        geno_melt = pl.concat([geno_melt, hom])

    var_pheno_df = geno_melt.lazy().join(melted_pheno_df, on='individual', how='left')
    if debug:
        # Return intermediate dataframe if debugging
        return var_pheno_df
    
    var_pheno_df = var_pheno_df.group_by(
           ['id', 'phenotype']
           ).agg([
                pl.len().alias('n_individuals'),
                pl.col('pheno_value').mean().cast(pl.Float32).alias('mean_pheno_value'),
                pl.col('pheno_value').std().cast(pl.Float32).alias('std_pheno_value'),
            ]).drop_nulls(subset=['mean_pheno_value'])
    return var_pheno_df

In [16]:
output_dir = "/home/dnanexus/data_dir/var_pheno_EUR_chunks"
chunk_size = 10_000

for chunk_num in tqdm(range(var_ids.shape[0]//chunk_size + 1)):
    tmp = process_genotype_chunk(
        geno=geno[chunk_num*chunk_size:(chunk_num+1)*chunk_size, olink_indices],
        var_ids=var_ids[chunk_num*chunk_size:(chunk_num+1)*chunk_size],
        sample_list=sample_ids[olink_indices],
        melted_pheno_df=olink_melt.lazy(),
        homozygous=False,
    ).collect()
    break

tmp

  0%|          | 0/183 [00:17<?, ?it/s]


id,phenotype,n_individuals,mean_pheno_value,std_pheno_value
str,str,u64,f32,f32
"""chr1:3467169:C:G""","""ENSG00000081800_olink""",1,-0.363691,null
"""chr1:2229364:A:G""","""ENSG00000135218_olink""",1,0.408683,null
"""chr1:8967768:C:T""","""ENSG00000082074_olink""",7,0.198808,1.265288
"""chr1:6213135:C:T""","""ENSG00000065361_olink""",4,0.381372,1.242684
"""chr1:3480477:GGA:G""","""ENSG00000124678_olink""",1,1.095063,null
…,…,…,…,…
"""chr1:2519307:T:C""","""ENSG00000168769_olink""",1,-0.429507,null
"""chr1:1374025:T:C""","""ENSG00000019991_olink""",1496,-0.004606,1.019444
"""chr1:10105685:T:C""","""ENSG00000120889_olink""",1,-0.988386,null


In [17]:
tmp['id'].value_counts(sort=True)

id,count
str,u64
"""chr1:2229364:A:G""",371
"""chr1:8967768:C:T""",371
"""chr1:6213135:C:T""",371
"""chr1:3480477:GGA:G""",371
"""chr1:2519857:C:T""",371
…,…
"""chr1:11059839:C:A""",123
"""chr1:2518220:C:T""",99
"""chr1:11056741:G:C""",63


In [18]:
geno_clipped = _fast_clip_and_sum_allels(geno[:10_000, olink_indices])
geno_clipped

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(10000, 40398), dtype=int8)

In [25]:
sum(geno_clipped.sum(axis=1) > 0)

np.int64(2338)

In [19]:
rows, cols = np.where(geno_clipped == 1)

In [20]:
sample_list=sample_ids[olink_indices]
geno_melt = pl.DataFrame({
        'id': var_ids[rows],
        'individual': sample_list[cols],
        'genotype': 1
    })

geno_melt

id,individual,genotype
str,str,i32
"""chr1:1373805:T:C""","""3622589""",1
"""chr1:1373805:T:C""","""3540179""",1
"""chr1:1373805:T:C""","""3991045""",1
"""chr1:1373805:T:C""","""3130113""",1
"""chr1:1373805:T:C""","""4471351""",1
…,…,…
"""chr1:11231009:G:A""","""2259790""",1
"""chr1:11231013:G:A""","""2141582""",1
"""chr1:11231040:T:C""","""2938277""",1


In [21]:
var_pheno_df = geno_melt.join(olink_melt, on='individual', how='left')
var_pheno_df

id,individual,genotype,phenotype,pheno_value
str,str,i32,str,f64
"""chr1:1373805:T:C""","""3622589""",1,"""ENSG00000120053_olink""",-1.273965
"""chr1:1373805:T:C""","""3622589""",1,"""ENSG00000140443_olink""",1.564473
"""chr1:1373805:T:C""","""3622589""",1,"""ENSG00000084674_olink""",-0.164796
"""chr1:1373805:T:C""","""3622589""",1,"""ENSG00000015475_olink""",-1.803142
"""chr1:1373805:T:C""","""3622589""",1,"""ENSG00000117748_olink""",-0.09892
…,…,…,…,…
"""chr1:11231048:G:A""","""3611469""",1,"""ENSG00000124678_olink""",0.442166
"""chr1:11231048:G:A""","""3611469""",1,"""ENSG00000109062_olink""",1.012175
"""chr1:11231048:G:A""","""3611469""",1,"""ENSG00000167186_olink""",0.72797


In [23]:
var_pheno_df['id'].value_counts(sort=True)

id,count
str,u64
"""chr1:8949385:A:C""",7447825
"""chr1:8949347:C:T""",7087584
"""chr1:2512975:G:A""",7044548
"""chr1:6218354:A:G""",6735134
"""chr1:8957145:A:G""",6316646
…,…
"""chr1:11231009:G:A""",371
"""chr1:11231013:G:A""",371
"""chr1:11231040:T:C""",371


In [22]:
var_pheno_df.group_by(
           ['id', 'phenotype']
           ).agg([
                pl.len().alias('n_individuals'),
                pl.col('pheno_value').mean().cast(pl.Float32).alias('mean_pheno_value'),
                pl.col('pheno_value').std().cast(pl.Float32).alias('std_pheno_value'),
            ]).drop_nulls(subset=['mean_pheno_value'])

id,phenotype,n_individuals,mean_pheno_value,std_pheno_value
str,str,u64,f32,f32
"""chr1:2303064:G:A""","""ENSG00000179364_olink""",3,0.233165,0.055269
"""chr1:6219305:G:C""","""ENSG00000159202_olink""",2,0.41997,2.657589
"""chr1:10137198:C:T""","""ENSG00000259384_olink""",1,2.07007,null
"""chr1:10147066:G:A""","""ENSG00000149131_olink""",1,-0.132662,null
"""chr1:10168247:G:A""","""ENSG00000078018_olink""",2,0.96364,3.645902
…,…,…,…,…
"""chr1:11055829:G:A""","""ENSG00000161911_olink""",1,0.86707,null
"""chr1:3707720:G:A""","""ENSG00000177628_olink""",1,2.786338,null
"""chr1:2510755:A:G""","""ENSG00000101439_olink""",2,-0.227494,1.860898
